In [1]:
from utils.utils import load_CSIRO, CSIRO_group_k_fold, CSIRO_stratified_group_k_fold
from torchvision.transforms import v2
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
from rich.console import Console
from rich.table import Table


In [2]:
r2_coeff = {
    "Dry_Green_g": 0.1,
    "Dry_Clover_g": 0.1,
    "Dry_Dead_g": 0.1,
    "Dry_Total_g": 0.5,
    "GDM_g": 0.2
}
input_h = 768
input_w = 1536
gating = False
hidden_dim = 512
model_names = {
    "Dry_Green_g": "facebook/dinov3-vits16plus-pretrain-lvd1689m",
    "Dry_Clover_g": "facebook/dinov3-vits16plus-pretrain-lvd1689m",
    "Dry_Dead_g": "facebook/dinov3-vitb16-pretrain-lvd1689m"
}
weight_paths = {
    "Dry_Green_g": "../wandb/20260119_1327_CSIRO/wandb/run-20260119_132732-5aximi38/files/0_best_model_0.725.pth",
    "Dry_Clover_g": "../wandb/20260119_1640_CSIRO/wandb/run-20260119_164056-kec6cn4p/files/0_best_model_0.716.pth",
    "Dry_Dead_g": "../wandb/20260119_0657_ViT_0/0_best_model_0.482.pth"
}
predict_heights = {
    "Dry_Green_g": True,
    "Dry_Clover_g": False,
    "Dry_Dead_g": True
}

In [3]:
def compute_global_mean(dataset):
    global_means = {k: 0 for k in r2_coeff.keys()}
    for row in dataset.data_values:
        for target_name, _ in r2_coeff.items():
            global_means[target_name] += row[target_name]
    for k in global_means.keys():
        global_means[k] /= len(dataset.data_values)

    return global_means

In [4]:
def inference(dataloader, model, target_class):
    # Main dictionary to store results keyed by image path
    results = {}

    for data_dict in tqdm(dataloader):
        # Move all tensors to the configured device
        for k, v in data_dict.items():
            if isinstance(v, torch.Tensor):
                data_dict[k] = v.to(device)

        # Standard inference inputs (Multi-scale logic removed)
        input_imgs = data_dict["Input_Img"]

        # Run Model
        if gating:
            pred_dict = model(input_imgs, return_patch_preds=True, return_gates=True)
        else:
            pred_dict = model(input_imgs, return_patch_preds=True)

        # Iterate over the batch to map data to specific image paths
        batch_paths = data_dict["image_path"]
        batch_size = input_imgs.shape[0]

        for i in range(batch_size):
            # Ensure the path is a string to use as a dictionary key
            path = str(batch_paths[i])

            # Initialize the entry for this specific image
            results[path] = {
                "predictions": None,
                "targets": None,
                "patch_predictions": None,
                "gate_predictions": None,
                "Height_Ave_cm": None
            }

            # Pred and target
            results[path]["predictions"] = pred_dict[target_class][i].detach().cpu()
            results[path]["targets"] = data_dict[target_class][i].detach().cpu()

            # Patch prediction
            tile_key = f"Tile_{target_class}"
            if tile_key in pred_dict:
                results[path]["patch_predictions"] = pred_dict[tile_key][i].detach().cpu()

            # Extract Gates if enabled
            if gating:
                gate_key = f"Tile_Gate_{k}"
                if gate_key in pred_dict:
                    results[path]["gate_predictions"] = pred_dict[gate_key][i].detach().cpu()

             # 3. Extract Height (Optional)
            if predict_height and "Height_Ave_cm" in pred_dict:
                results[path]["Height_Ave_cm"] = pred_dict["Height_Ave_cm"][i].detach().cpu()

    return results

In [5]:
from dataset import CSIRODataset
from model.DinoV3ViT import DinoV3ViT

data_folder = "../data/CSIRO"
df = load_CSIRO(data_folder)
batch_size = 4
device = "cuda"
train_idxs, val_idxs = CSIRO_stratified_group_k_fold(df)
train_df = df.iloc[train_idxs[0]]
val_df = df.iloc[val_idxs[0]]
train_dataset = CSIRODataset(data_folder, train_df, input_h, input_w, False, False)
val_dataset = CSIRODataset(data_folder, val_df, input_h, input_w, False, False)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
train_global_means = compute_global_mean(train_dataset)
val_global_means = compute_global_mean(val_dataset)

train_results = {}
val_results = {}

for target_class in ["Dry_Green_g", "Dry_Clover_g", "Dry_Dead_g"]:
    print(f"Predicting {target_class}")
    model = DinoV3ViT(
        model_name=model_names[target_class],
        hidden_dim=hidden_dim,
        split_img=False,
        predict_height=predict_heights[target_class]
    )

    model.load_state_dict(torch.load(weight_paths[target_class], weights_only=True))
    model.to(device)
    model.eval()

    with torch.no_grad():
        cur_train_results = inference(train_dataloader, model, target_class)
        cur_val_results = inference(val_dataloader, model, target_class)
    del model

    for image_path in cur_train_results:
        if image_path not in train_results:
            train_results[image_path] = {
                "predictions": {},
                "targets": {},
                "patch_predictions": {},
                "gate_predictions": {},
                "Height_Ave_cm": {}
            }

        for k in cur_train_results[image_path]:
            train_results[image_path][k][target_class] = cur_train_results[image_path][k]
            if k in ["predictions", "targets"]:
                train_results[image_path][k]["Dry_Total_g"] = train_results[image_path][k].get("Dry_Total_g", 0) + cur_train_results[image_path][k]
                if target_class != "Dry_Dead_g":
                    train_results[image_path][k]["GDM_g"] = train_results[image_path][k].get("GDM_g", 0) + cur_train_results[image_path][k]

    for image_path in cur_val_results:
        if image_path not in val_results:
            val_results[image_path] = {
                "predictions": {},
                "targets": {},
                "patch_predictions": {},
                "gate_predictions": {},
                "Height_Ave_cm": {}
            }

        for k in cur_val_results[image_path]:
            val_results[image_path][k][target_class] = cur_val_results[image_path][k]
            if k in ["predictions", "targets"]:
                val_results[image_path][k]["Dry_Total_g"] = val_results[image_path][k].get("Dry_Total_g", 0) + cur_val_results[image_path][k]
                if target_class != "Dry_Dead_g":
                    val_results[image_path][k]["GDM_g"] = val_results[image_path][k].get("GDM_g", 0) + cur_val_results[image_path][k]

Predicting Dry_Green_g


  0%|          | 0/73 [00:02<?, ?it/s]


NameError: name 'predict_height' is not defined

In [ ]:
def compute_r2(results, weighted_mean):
    total_weighted_r2 = 0
    stats = {}

    # Temporary lists to aggregate data for global calculation
    agg_preds = {k: [] for k in r2_coeff.keys()}
    agg_targets = {k: [] for k in r2_coeff.keys()}

    # 1. Iterate through the results dictionary to compute per-image errors
    for path, data in results.items():
        # Initialize a sub-dictionary for errors if it doesn't exist
        data["errors"] = {}

        for k in r2_coeff.keys():
            # Retrieve tensor values (already on CPU from inference)
            pred = data["predictions"][k]
            target = data["targets"][k]

            # Collect for global stats
            agg_preds[k].append(pred)
            agg_targets[k].append(target)

            # Compute Squared Error for this specific image and attach it
            # We use .item() to store it as a standard float for easier reading later
            error_val = (target - pred) ** 2
            data["errors"][k] = error_val.item()

    # 2. Compute Global Stats (R2, MSE, Max Error)
    for k in r2_coeff.keys():
        # Stack the aggregated lists into tensors for vectorized calculation
        flat_preds = torch.stack(agg_preds[k])
        flat_targets = torch.stack(agg_targets[k])

        # Calculate MSE
        mse = torch.sum((flat_targets - flat_preds) ** 2)

        # Calculate Variance (using the provided weighted_mean)
        var = torch.sum((flat_targets - weighted_mean[k]) ** 2)

        # Calculate R2
        r2 = 1 - mse / var
        total_weighted_r2 += r2_coeff[k] * r2

        # Calculate error stats
        errors = (flat_targets - flat_preds) ** 2
        stats[k] = {
            "r2": r2.item(),
            "mse": errors.mean().item(),
            "max": errors.max().item()
        }

    return total_weighted_r2, stats, results

In [ ]:
train_r2, train_stats, train_results = compute_r2(train_results, train_global_means)
val_r2, val_stats, val_image_scores = compute_r2(val_results, val_global_means)
# 2. Build the Master Table
console = Console()
table = Table(title="CSIRO Model Performance", show_header=True, header_style="bold magenta")

# Define Columns
table.add_column("Target", style="cyan", no_wrap=True)
table.add_column("Train R2", justify="right", style="green")
table.add_column("Val R2", justify="right", style="green")
table.add_column("Train MSE", justify="right", style="yellow")
table.add_column("Val MSE", justify="right", style="yellow")
# Optional: Add Max Error if you want detail
# table.add_column("Val MaxErr", justify="right", style="red")

# 3. Add Rows for each Target
for target in r2_coeff.keys():
    t_dat = train_stats[target]
    v_dat = val_stats[target]

    # Highlight overfitting (if Val R2 is significantly lower than Train R2)
    r2_color = "red" if (t_dat['r2'] - v_dat['r2']) > 0.1 else "green"

    table.add_row(
        target,
        f"{t_dat['r2']:.4f}",
        f"[{r2_color}]{v_dat['r2']:.4f}[/]",  # Conditional coloring
        f"{t_dat['mse']:.4f}",
        f"{v_dat['mse']:.4f}",
    )

# 4. Add Summary Row (Weighted R2)
table.add_section()
table.add_row(
    "[bold]Weighted Score[/]",
    f"[bold]{train_r2:.4f}[/]",
    f"[bold]{val_r2:.4f}[/]",
    "-", "-"
)
console.print(table)

In [ ]:
import matplotlib.pyplot as plt
import torch
import os
import math
from torchvision.transforms import v2
from torchvision.io import read_image

# Note: Ensure 'gating', 'input_h', 'input_w', 'predict_height' are defined globally

def visualize_patch(image_path, target_class, img_data, ax_img, ax_map, ax_gate, height, species, input_h, input_w, split_img=True, data_folder="../data/CSIRO"):
    """
    Draws the image, prediction heatmap for 'target_class', and detailed stats for all classes.
    """
    # --- 1. Image Processing ---
    img_full_path = image_path
    if not os.path.exists(img_full_path):
        ax_img.text(0.5, 0.5, "Img Not Found", ha='center', va='center')
        ax_img.axis("off")
        ax_map.axis("off")
        if ax_gate: ax_gate.axis("off")
        return

    img = read_image(img_full_path)
    full_img = v2.Resize((input_h, input_w), antialias=True)(img).permute(1, 2, 0)
    num_splits = 1

    # --- 2. Prepare Maps (For the specific target_class) ---
    tile_key = f"{target_class}"
    patch_pred = img_data["patch_predictions"][tile_key]

    grid_h = int(input_h / 16)
    grid_w = int(input_w / 16)

    # Reshape: (num_splits, grid_h, grid_w)
    reshape_pred = patch_pred.reshape(num_splits, grid_h, grid_w)

    if split_img:
        pred_map = torch.cat([reshape_pred[0], reshape_pred[1]], dim=1)
    else:
        pred_map = reshape_pred[0]

    # Handle Gating
    gate_map = None
    if gating and ax_gate is not None:
        gate_key = f"Tile_Gate_{target_class}"
        if gate_key in img_data["patch_predictions"]:
            gate_pred = img_data["patch_predictions"][gate_key]
            reshape_gate = gate_pred.reshape(num_splits, grid_h, grid_w)
            if split_img:
                gate_map = torch.cat([reshape_gate[0], reshape_gate[1]], dim=1)
            else:
                gate_map = reshape_gate[0]

    # --- 3. Plotting ---

    # Row 1: Real Image
    ax_img.imshow(full_img)
    ax_img.axis("off")
    ax_img.set_title(f"Input: {species}", fontsize=9)

    # Row 2: Prediction Heatmap
    im_pred = ax_map.imshow(pred_map.float(), cmap="jet", interpolation="nearest")
    ax_map.axis("off")
    plt.colorbar(im_pred, ax=ax_map, fraction=0.046, pad=0.04)

    # --- Constructing the Multi-Class Title (UPDATED) ---
    preds = img_data["predictions"]
    targets = img_data["targets"]
    current_err = img_data["errors"][target_class]

    fmt = lambda x: f"{x:.2f}"

    # Mapping full keys to short labels
    class_map = {
        "Dry_Green_g": "Green",
        "Dry_Clover_g": "Clover",
        "Dry_Dead_g": "Dead"
    }

    # 1. Format the PRIORITY class (Target Class)
    # This puts the specific class error and values on the first line
    t_label = class_map.get(target_class, "?")
    t_pred = fmt(preds[target_class].item())
    t_targ = fmt(targets[target_class].item())

    # Using brackets or symbols to make it pop
    line_primary = f"Err: {current_err:.3f}  |  [{t_label}]: {t_pred} / {t_targ}"

    # 2. Format the OTHER classes
    other_texts = []
    for k, label in class_map.items():
        if k != target_class:
            p_val = fmt(preds[k].item())
            t_val = fmt(targets[k].item())
            other_texts.append(f"{label}: {p_val}/{t_val}")

    # 3. Add Height to the 'Others' list
    if predict_heights[target_class]:
         h_txt = f"H: {fmt(img_data['Height_Ave_cm'][target_class].item())}/{fmt(height)}"
    else:
         h_txt = f"H: {height}"
    other_texts.append(h_txt)

    # Join the secondary line
    line_secondary = "  ".join(other_texts)

    # Final Title Construction
    stats_text = f"{line_primary}\n{line_secondary}"

    ax_map.set_title(stats_text, fontsize=8, color='darkred', fontweight='bold')

    # Row 3: Gate Heatmap
    if gating and gate_map is not None:
        im_gate = ax_gate.imshow(gate_map.float(), cmap="magma", interpolation="nearest")
        ax_gate.axis("off")
        plt.colorbar(im_gate, ax=ax_gate, fraction=0.046, pad=0.04)
        ax_gate.set_title("Gate Attention", fontsize=9)


def visualize(results, data_df, num_show=3, split_img=True, max_cols=5):
    """
    Visualizes worst performing images based on the 'results' dictionary.
    """
    # 1. Prepare Data Lookups from DataFrame (for static info like Species/Height)
    image_path_list = data_df["image_path"].tolist()

    # Handle Height: if present in DF use it, otherwise default 0
    if "Height_Ave_cm" in data_df.columns:
        height_list = data_df["Height_Ave_cm"].tolist()
    else:
        height_list = [0.0] * len(image_path_list)

    species_list = data_df["Species"].tolist()

    image_2_height = {path: h for path, h in zip(image_path_list, height_list)}
    image_2_species = {path: s for path, s in zip(image_path_list, species_list)}

    # 2. Iterate over the 3 classes
    for target_class in ["Dry_Green_g", "Dry_Clover_g", "Dry_Dead_g"]:

        # --- A. SORTING LOGIC ---
        # Create a list of (image_path, error) tuples for the current class
        # results[path]["errors"] is a dict { "Dry_Green_g": 0.05, ... }
        ranked_images = []
        for path, data in results.items():
            if target_class in data["errors"]:
                ranked_images.append( (path, data["errors"][target_class]) )

        # Sort descending by error
        ranked_images.sort(key=lambda x: x[1], reverse=True)

        # Select top N worst
        worst_batch = ranked_images[:num_show]

        if len(worst_batch) == 0:
            continue

        # --- B. BATCHING & PLOTTING ---
        total_batches = math.ceil(len(worst_batch) / max_cols)

        for batch_idx in range(total_batches):
            start_i = batch_idx * max_cols
            end_i = start_i + max_cols
            current_batch = worst_batch[start_i:end_i]

            n_cols = len(current_batch)
            n_rows = 3 if gating else 2

            fig, axes = plt.subplots(nrows=n_rows, ncols=n_cols,
                                     figsize=(4 * n_cols, 6),
                                     squeeze=False)

            fig.suptitle(f"Worst {target_class} (Batch {batch_idx+1}/{total_batches})", fontsize=16)

            for i, (img_path, error_val) in enumerate(current_batch):

                # Retrieve Full Data for this image
                img_data = results[img_path]

                cur_height = image_2_height.get(f"train/{os.path.basename(img_path)}", 0.0)
                cur_species = image_2_species.get(f"train/{os.path.basename(img_path)}", "Unknown")

                visualize_patch(
                    image_path=img_path,
                    target_class=target_class, # Pass class to decide which heatmap to show
                    img_data=img_data,         # Pass full data to show all scalar stats
                    ax_img=axes[0, i],
                    ax_map=axes[1, i],
                    ax_gate=axes[2, i] if gating else None,
                    height=cur_height,
                    species=cur_species,
                    input_h=input_h,
                    input_w=input_w,
                    split_img=split_img
                )

            # Hide empty columns if last batch is incomplete
            if n_cols < max_cols and total_batches > 1:
                # This part is tricky with squeeze=False dynamic sizing,
                # but usually matplotlib handles empty space fine.
                # Explicitly turning off unused axes is cleaner but optional here.
                pass

            plt.tight_layout()
            plt.show()

# --- Example Usage ---

# print("Visualizing Validation Set (Worst 5 per class):")
visualize(val_results, val_df, num_show=50, split_img=False, max_cols=2)

In [ ]:
del model
del df
del train_dataset
del train_dataloader
del val_dataset
del val_dataloader

| Metric | Model 1 Train | Model 1 Val | Model 2 Train | Model 2 Val | Model 3 Train | Model 3 Val | Model 4 Train | Model 4 Val |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **Main Loss** | 3.4208 | 4.4226 | 2.1489 | 4.1754 | 4.9879 | 7.0076 | 2.0152 | 10.3557 |
| **R2 Score (Weighted)** | 0.8459 | 0.4489 | 0.9232 | 0.7631 | 0.6854 | 0.6125 | 0.8947 | 0.5191 |
| **Dry_Green_g R2** | 0.9118 | 0.3420 | 0.9583 | 0.8377 | 0.7799 | 0.7800 | 0.9570 | 0.4910 |
| **Dry_Clover_g R2** | 0.8787 | 0.4508 | 0.9029 | 0.8883 | 0.6497 | 0.7142 | 0.9486 | 0.4539 |
| **Dry_Dead_g R2** | 0.6684 | 0.3581 | 0.8451 | 0.5658 | 0.5267 | 0.2758 | 0.7464 | -0.0070 |
| **Dry_Total_g R2** | 0.8450 | 0.4790 | 0.9274 | 0.7263 | 0.6840 | 0.5661 | 0.8801 | 0.6983 |
| **GDM_g R2** | 0.8874 | 0.4716 | 0.9442 | 0.8540 | 0.7387 | 0.7622 | 0.9472 | 0.3809 |